# ViaAlerta — detecção de buracos

Pipeline de uma classe: `0 = buraco`. Prepara as anotações, confere a integridade,
treina na GPU e apresenta curvas de aprendizado e avaliação final.

O ponto de partida é o detector atual de buracos. A busca genética é opcional e
fica desativada por padrão para priorizar o prazo de entrega. Os experimentos
anteriores são preservados e a publicação na Azure é uma etapa posterior.

## 1. Configuração

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import torch

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'api/app.py').is_file() and (p / 'notebooks').is_dir()), None)
if PROJECT is None:
    raise RuntimeError('Abra o notebook dentro de ViaAlerta_Projeto.')
RUN = datetime.now().strftime('%Y%m%d-%H%M%S')
SOURCE_DATASET = PROJECT / 'notebooks/dados/viaalerta_multiclasse_v1'
DATASET = PROJECT / f'notebooks/dados/viaalerta_buracos_{RUN}'
OUTPUT = PROJECT / f'notebooks/treinamentos/buracos_finetune_{RUN}'
BASELINE = PROJECT / 'notebooks/treinamentos/buracos_aug_nativo_01/yolo_300ep-3/weights/best.pt'
WEIGHTS = BASELINE
BUSCA_YOLO = False
TREINAR_YOLO = True
EPOCHS = 120
BATCH = 32
IMGSZ = 640
DEVICE = '0'
YOLO_SEARCH_TRIALS = 4
YOLO_SEARCH_EPOCHS = 5
if not torch.cuda.is_available():
    raise RuntimeError('CUDA indisponível. Selecione o ambiente de treinamento com GPU.')
if not WEIGHTS.is_file():
    raise FileNotFoundError(f'Detector inicial ausente: {WEIGHTS}')
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', DATASET)
print('Saída:', OUTPUT)


## 2. Preparação e integridade

As imagens vêm do manifesto já revisado. Conservamos as divisões de treino,
validação e teste, inclusive os holdouts antigos. Copiamos as fotos para uma nova
base e mantemos somente caixas de buracos. Imagens anotadas apenas com outras
classes são excluídas; elas não são transformadas automaticamente em negativos.
Os negativos já existentes são preservados. O OpenCV confere a leitura das imagens.

In [ ]:
from collections import Counter, defaultdict
import hashlib
import math
import shutil
import cv2
import numpy as np
import yaml

def hash_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def caixas_buracos(text):
    boxes = []
    total = 0
    for line in text.splitlines():
        if not line.strip():
            continue
        values = list(map(float, line.split()))
        if len(values) != 5 or not all(math.isfinite(v) for v in values):
            raise ValueError('Anotação inválida na base de origem.')
        cls, x, y, w, h = values
        if cls != int(cls) or cls < 0 or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError('Classe ou coordenadas inválidas.')
        if min(x-w/2, y-h/2) < -1e-7 or max(x+w/2, y+h/2) > 1.0000001:
            raise ValueError('Caixa fora da imagem.')
        total += 1
        if cls == 0:
            boxes.append([0, x, y, w, h])
    return boxes, total

def preparar_buracos(source, destination):
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if destination.exists():
        raise FileExistsError('Use uma saída nova para preservar a base existente.')
    config = yaml.safe_load((source/'data.yaml').read_text(encoding='utf-8'))
    names = config['names']
    if (names[0] if isinstance(names, list) else names.get(0, names.get('0'))) != 'buraco':
        raise ValueError('A classe zero da origem precisa ser buraco.')
    rows = [json.loads(line) for line in (source/'manifest.jsonl').read_text(encoding='utf-8').splitlines()]
    staging = destination.with_name(destination.name + '.preparando')
    staging.mkdir(parents=True, exist_ok=False)
    (staging/'images').mkdir()
    (staging/'labels').mkdir()
    selected, excluded = [], Counter()
    for row in rows:
        if not row['selected_for_detection'] or row['review_reasons']:
            excluded['pendentes_ou_nao_selecionadas'] += 1
            continue
        boxes, total = caixas_buracos((source/'labels'/(row['id']+'.txt')).read_text(encoding='utf-8'))
        if not boxes and total:
            excluded['somente_outras_classes'] += 1
            continue
        original = Path(row['image'])
        expected = row.get('output_sha256', row['sha256'])
        if hash_file(original) != expected:
            raise ValueError(f'Imagem de origem alterada: {original}')
        filename = row['id'] + original.suffix.lower()
        copied = staging/'images'/filename
        shutil.copy2(original, copied)
        new = {**row, 'image':str(destination/'images'/filename), 'boxes':boxes,
               'weak_visual_severity':[], 'projection':'buraco_only_v1'}
        (staging/'labels'/(row['id']+'.txt')).write_text(
            '\n'.join(' '.join(map(str, b)) for b in boxes), encoding='utf-8')
        selected.append(new)
        if len(selected) % 2000 == 0:
            print('Fotos preparadas:', len(selected), flush=True)
    if not selected:
        raise ValueError('Nenhuma imagem válida de buracos ou negativo conhecido.')
    for split in ['train','val','test']:
        items = [r['image'] for r in selected if r['split'] == split]
        if not items:
            raise ValueError(f'Divisão vazia: {split}')
        (staging/(split+'.txt')).write_text('\n'.join(items)+'\n',encoding='utf-8')
    (staging/'manifest.jsonl').write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in selected)+'\n',encoding='utf-8')
    (staging/'data.yaml').write_text(yaml.safe_dump({'path':str(destination),'train':'train.txt','val':'val.txt','test':'test.txt','names':{0:'buraco'}},sort_keys=False),encoding='utf-8')
    report = {'source':str(source),'class_names':['buraco'], 'images_by_split':dict(Counter(r['split'] for r in selected)),
              'boxes_by_split':dict(Counter({split:sum(len(r['boxes']) for r in selected if r['split']==split) for split in ['train','val','test']})),
              'negative_images':sum(not r['boxes'] for r in selected),'excluded':dict(excluded)}
    (staging/'preparation.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    staging.rename(destination)
    return report

def validar_buracos(dataset):
    dataset = Path(dataset)
    config = yaml.safe_load((dataset/'data.yaml').read_text(encoding='utf-8'))
    if config['names'] != {0:'buraco'}:
        raise ValueError('A base deve conter exclusivamente a classe buraco.')
    rows = [json.loads(line) for line in (dataset/'manifest.jsonl').read_text(encoding='utf-8').splitlines()]
    ids, pixels, groups = set(), defaultdict(set), defaultdict(set)
    for i, row in enumerate(rows,1):
        if row['id'] in ids:
            raise ValueError('ID repetido no manifesto.')
        ids.add(row['id'])
        image = Path(row['image'])
        if hash_file(image) != row.get('output_sha256',row['sha256']):
            raise ValueError(f'Cópia alterada: {image}')
        if hash_file(row['source_path']) != row.get('source_sha256',row['sha256']):
            raise ValueError(f'Arquivo original alterado: {row["id"]}')
        decoded = cv2.imdecode(np.fromfile(image,dtype=np.uint8),cv2.IMREAD_COLOR)
        if decoded is None:
            raise ValueError(f'OpenCV não conseguiu ler: {image}')
        boxes,total = caixas_buracos((dataset/'labels'/(row['id']+'.txt')).read_text(encoding='utf-8'))
        if len(boxes) != total or boxes != row['boxes']:
            raise ValueError('Classe diferente de buraco ou manifesto divergente.')
        pixels[row['pixel_sha']].add(row['split'])
        groups[row['group']].add(row['split'])
        if row['source'] == 'existing' and row['source_split'] in ['validacao','teste']:
            expected = {'validacao':'val','teste':'test'}[row['source_split']]
            if row['split'] != expected:
                raise ValueError('Holdout antigo mudou de divisão.')
        if i % 2000 == 0:
            print('Integridade conferida:',i,flush=True)
    if any(len(v)>1 for v in [*pixels.values(),*groups.values()]):
        raise ValueError('Grupo ou pixels compartilhados entre divisões.')
    for split in ['train','val','test']:
        listed = (dataset/(split+'.txt')).read_text(encoding='utf-8').splitlines()
        expected = [r['image'] for r in rows if r['split']==split]
        if listed != expected:
            raise ValueError('Lista divergente do manifesto.')
    report = {'passed':True,'images_verified':len(rows),'class_names':['buraco']}
    (dataset/'validation.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    return report


## 3. Testes rápidos antes do treinamento

In [ ]:
import tempfile

assert caixas_buracos('0 0.5 0.5 0.2 0.2\n1 0.5 0.5 0.1 0.1')[0] == [[0,0.5,0.5,0.2,0.2]]
assert caixas_buracos('1 0.5 0.5 0.1 0.1') == ([],1)
assert caixas_buracos('') == ([],0)
for invalid in ['0 nan 0.5 0.1 0.1','0 0.99 0.5 0.5 0.1','0 0.5 0.5 0 0.1','0.5 0.5 0.5 0.1 0.1']:
    try:
        caixas_buracos(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Anotação inválida aceita: {invalid}')
print('Filtro de classes, negativos e coordenadas: aprovado.')


## 4. Preparar e validar a base de uma classe

In [ ]:
preparation = preparar_buracos(SOURCE_DATASET, DATASET)
print(json.dumps(preparation,ensure_ascii=False,indent=2))
validation = validar_buracos(DATASET)
print(json.dumps(validation,indent=2))


## 5. Curvas de aprendizado

As curvas mostram treino e validação por época. A evolução do mAP e do recall
ajuda a avaliar se a redução da perda de treino também melhora a generalização.
O gráfico é salvo como `curvas_aprendizado.png` na pasta do detector.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def gerar_curvas(csv_path):
    csv_path = Path(csv_path)
    frame = pd.read_csv(csv_path)
    frame.columns = frame.columns.str.strip()
    if frame.empty:
        raise ValueError('Ainda não há épocas completas para desenhar as curvas.')
    figure, axes = plt.subplots(2,3,figsize=(16,9),constrained_layout=True)
    panels = [
        ('Perda das caixas', ['train/box_loss','val/box_loss'],['Treino','Validação']),
        ('Perda de classificação', ['train/cls_loss','val/cls_loss'],['Treino','Validação']),
        ('Perda DFL', ['train/dfl_loss','val/dfl_loss'],['Treino','Validação']),
        ('mAP na validação', ['metrics/mAP50(B)','metrics/mAP50-95(B)'],['mAP50','mAP50–95']),
        ('Precisão e recall na validação', ['metrics/precision(B)','metrics/recall(B)'],['Precisão','Recall']),
        ('Taxa de aprendizado', ['lr/pg0'],['Learning rate']),
    ]
    for ax,(title,columns,labels) in zip(axes.flat,panels):
        for column,label in zip(columns,labels):
            ax.plot(frame['epoch'],frame[column],label=label,linewidth=1.7)
        ax.set(title=title,xlabel='Época')
        ax.grid(alpha=0.25)
        ax.legend()
    figure.suptitle('ViaAlerta — aprendizado do detector de buracos',fontsize=17)
    target = csv_path.parent/'curvas_aprendizado.png'
    figure.savefig(target,dpi=170)
    plt.close(figure)
    return target


## 6. Treinamento e avaliação final

O ajuste fino parte do `best.pt` atual, com taxa inicial de aprendizado 0,001,
até 60 épocas e paciência de 12 épocas. Se habilitada, a busca genética seleciona
parâmetros pela validação. Ao final, o candidato e o modelo anterior são avaliados
no mesmo conjunto de teste, uma única vez por execução. Os resultados ficam em
`metrics.json`; nenhum peso é publicado automaticamente.

In [ ]:
from ultralytics import YOLO
from ultralytics.engine.tuner import Tuner
import math
import yaml

TUNABLE_PARAMETERS = {
    'lr0', 'lrf', 'momentum', 'weight_decay', 'warmup_epochs', 'warmup_momentum',
    'box', 'cls', 'cls_pw', 'dfl', 'hsv_h', 'hsv_s', 'hsv_v', 'degrees',
    'translate', 'scale', 'shear', 'perspective', 'flipud', 'fliplr', 'bgr',
    'mosaic', 'mixup', 'cutmix', 'copy_paste', 'close_mosaic',
}

def carregar_parametros_vencedores(path):
    path = Path(path)
    if not path.is_file():
        raise RuntimeError('A busca não gerou parâmetros vencedores. Confira os erros das tentativas.')
    parameters = yaml.safe_load(path.read_text(encoding='utf-8'))
    if not isinstance(parameters, dict) or not parameters:
        raise ValueError('Arquivo de parâmetros vencedores vazio ou inválido.')
    for key, value in parameters.items():
        if key not in TUNABLE_PARAMETERS:
            raise ValueError(f'Parâmetro inesperado no resultado da busca: {key}')
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            raise ValueError(f'Valor inválido no parâmetro {key}.')
    return parameters

def treinar_detector(data, weights, output, epochs=60, batch=8, imgsz=640,
                     device='0', trials=0, search_epochs=5, train_final=True, resume_search=False):
    data, weights, output = Path(data).resolve(), Path(weights).resolve(), Path(output).resolve()
    if not data.is_file() or not weights.is_file():
        raise FileNotFoundError('Informe dados e pesos locais existentes.')
    if epochs < 1 or batch < 1 or trials < 0 or search_epochs < 1:
        raise ValueError('Confira épocas, tamanho do lote e tentativas.')
    if not train_final and not trials:
        raise ValueError('Selecione busca, treinamento final ou ambos.')
    if resume_search:
        if not trials or not (output/'busca/tune_results.ndjson').is_file():
            raise ValueError('Para retomar, informe a pasta que já contém busca/tune_results.ndjson.')
        if (output/'detector').exists() or (output/'metrics.json').exists():
            raise FileExistsError('Esta execução já iniciou o treino final. Não há retomada automática dessa etapa.')
        previous_args = list(output.glob('multitrain*/**/args.yaml'))
        if not previous_args:
            raise ValueError('Não foi possível conferir a configuração da busca interrompida.')
        for artifact in previous_args:
            previous = yaml.safe_load(artifact.read_text(encoding='utf-8'))
            expected = dict(data=str(data), batch=batch, imgsz=imgsz, epochs=search_epochs, optimizer='AdamW')
            for key, value in expected.items():
                if previous.get(key) != value:
                    raise ValueError(f'Configuração de retomada divergente: {key}')
    elif output.exists() and any(output.iterdir()):
        raise FileExistsError('Use uma saída nova para preservar os experimentos anteriores.')
    output.mkdir(parents=True, exist_ok=True)
    common = dict(data=str(data), batch=batch, imgsz=imgsz, device=device,
                  project=str(output), seed=42, workers=0, cache=False,
                  patience=12, lr0=0.001, val=True, split='val', optimizer='AdamW',
                  resume=False, save=True)
    winner = {}
    search_summary = None
    if trials:
        search_model = YOLO(str(weights))
        tuner = Tuner(args={**common, 'model': str(weights), 'mode': 'train',
                           'epochs': search_epochs, 'name': 'busca', 'plots': False, 'resume': resume_search},
                      model=search_model.model, _callbacks=search_model.callbacks)
        tuner(iterations=trials)
        winner_path = Path(tuner.tune_dir)/'best_hyperparameters.yaml'
        winner = carregar_parametros_vencedores(winner_path)
        log_path = Path(tuner.tune_dir)/'tune_results.ndjson'
        records = [json.loads(line) for line in log_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        valid = [r for r in records if r.get('datasets') and not r.get('failed_datasets')
                 and all(r['datasets'].values()) and math.isfinite(float(r.get('fitness', float('nan'))))]
        if not valid:
            raise RuntimeError('Nenhuma tentativa da busca produziu métricas válidas de validação.')
        selected = max(valid, key=lambda r: float(r['fitness']))
        # O YAML e o log devem descrever o mesmo candidato, inclusive após falhas parciais.
        logged_parameters = selected.get('hyperparameters', {})
        if set(winner) != set(logged_parameters) or any(
            not math.isclose(float(value), float(logged_parameters[key]), rel_tol=1e-5, abs_tol=1e-8)
            for key, value in winner.items()
        ):
            raise RuntimeError('Os parâmetros vencedores divergem do log da busca. Confira os artefatos antes de treinar.')
        search_summary = {'trials_requested': trials, 'trials_recorded': len(records),
                          'valid_trials': len(valid), 'epochs_per_trial': search_epochs,
                          'selection_split': 'val', 'best_hyperparameters_file': str(winner_path),
                          'best_parameters': winner, 'best_candidate': selected,
                          'optimizer': 'AdamW'}
        (output/'search_summary.json').write_text(json.dumps(search_summary, indent=2), encoding='utf-8')
        del tuner, search_model
    if not train_final:
        return {'search': search_summary, 'final_training_started': False}
    final_parameters = {**common, **winner, 'epochs': epochs, 'name': 'detector', 'plots': True}
    (output/'final_parameters.json').write_text(json.dumps({
        'genetic_parameters_applied': bool(trials), 'initial_weights': str(weights),
        'parameters': final_parameters}, indent=2), encoding='utf-8')
    print('Parâmetros genéticos aplicados ao treinamento final:', winner if trials else 'busca não executada')
    model = YOLO(str(weights))
    model.add_callback('on_train_end', lambda trainer: gerar_curvas(Path(trainer.save_dir)/'results.csv'))
    result = model.train(**final_parameters)
    best = Path(model.trainer.best)
    if not best.is_file():
        raise RuntimeError('O treinamento final não gerou best.pt; o teste não será executado.')
    test = YOLO(str(best)).val(data=str(data), split='test', device=device,
                              imgsz=imgsz, batch=batch, workers=0,
                              project=str(output), name='teste_final')
    baseline_test = YOLO(str(BASELINE)).val(data=str(data), split='test', device=device,
                                              imgsz=imgsz, batch=batch, workers=0,
                                              project=str(output), name='teste_baseline')
    metrics = {'baseline_test': baseline_test.results_dict, 'same_test_dataset': str(data),
               'class_names': ['buraco'], 'search': search_summary, 'genetic_parameters_applied': bool(trials),
               'final_parameters_file': str(output/'final_parameters.json'),
               'best_weights': str(best), 'validation': result.results_dict,
               'test': test.results_dict, 'production_model_replaced': False}
    (output/'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return metrics


In [ ]:
if TREINAR_YOLO or BUSCA_YOLO:
    pipeline_results = treinar_detector(
        DATASET/'data.yaml', WEIGHTS, OUTPUT, epochs=EPOCHS, batch=BATCH,
        imgsz=IMGSZ, device=DEVICE, trials=YOLO_SEARCH_TRIALS if BUSCA_YOLO else 0,
        search_epochs=YOLO_SEARCH_EPOCHS, train_final=TREINAR_YOLO,
    )
else:
    print('Treinamento desativado na configuração.')


## 7. Gráfico e comparação dos resultados

In [ ]:
from IPython.display import display, Image

if (OUTPUT/'detector/results.csv').is_file():
    chart = gerar_curvas(OUTPUT/'detector/results.csv')
    display(Image(filename=str(chart)))
    print('Gráfico:', chart)
if (OUTPUT/'metrics.json').is_file():
    metrics = json.loads((OUTPUT/'metrics.json').read_text(encoding='utf-8'))
    columns = ['metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)','metrics/mAP50-95(B)']
    comparison = pd.DataFrame({
        'Anterior — mesmo teste': {k:metrics['baseline_test'][k] for k in columns},
        'Novo — mesmo teste': {k:metrics['test'][k] for k in columns},
    })
    display(comparison)
    print('Métricas:',OUTPUT/'metrics.json')
